# SmolVLA · перенос направления и сохранение навыков
Загрузите JSON-отчёты серверных скриптов: `*.direction.json`, `*.cka.json`, `paired.json` и JSON разбиения из `smolvla_skill_panel.py plan`. Локально укажите пути в `REPORT_PATHS`. Для графиков нужны только отчёты. Формирование оценок и порядок запуска описаны в `docs/research/CDPR_CHECKPOINT_RETENTION.md`.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

REPORT_PATHS = []  # Например, ["B.direction.json", "B_to_C.cka.json", "move/paired.json"].
if REPORT_PATHS:
    reports = {path: json.loads(Path(path).read_text()) for path in REPORT_PATHS}
else:
    from google.colab import files
    reports = {name: json.loads(blob) for name, blob in files.upload().items()}
print("Загружены:", ", ".join(reports))

In [ ]:
def plot_direction(report, partition="ordinary_test", denoising_step="first"):
    fig, ax = plt.subplots(figsize=(9, 4))
    for entry in report["comparisons"]:
        if entry["split"] != partition or "readouts" not in entry:
            continue
        readouts = entry["readouts"]
        layers = sorted(key for key in readouts if key.startswith("mlp_") and key.endswith(denoising_step))
        label = f'{entry["fit_skill"]} → {entry["eval_skill"]}'
        line, = ax.plot([int(key.split("_")[1]) for key in layers],
                        [readouts[key]["mean_cosine"] for key in layers],
                        "o-", label=label)
        ax.axhline(readouts["proprio"]["mean_cosine"], color=line.get_color(), linestyle=":", alpha=0.5)
    ax.set(xlabel="Слой action expert", ylabel="Средний cosine с направлением к цели",
           title=f"{partition} · выход MLP · {denoising_step}")
    if ax.get_legend_handles_labels()[0]:
        ax.legend()
    plt.show()

def plot_cka(report, denoising_step="first"):
    fig, ax = plt.subplots(figsize=(9, 4))
    for skill, values in report["linear_cka"].items():
        layers = sorted(key for key, value in values.items()
                        if key.startswith("mlp_") and key.endswith(denoising_step) and value is not None)
        ax.plot([int(key.split("_")[1]) for key in layers],
                [values[key] for key in layers], "o-", label=skill)
    ax.set(xlabel="Слой action expert", ylabel="Linear CKA",
           title=f"Одни и те же сцены между чекпоинтами · {denoising_step}", ylim=(0, 1.02))
    ax.legend()
    plt.show()

def plot_paired(report, instruction=None, depth=None):
    if depth:
        effects = {arm: skills[depth] for arm, skills in report["late_minus_early"].items() if depth in skills}
    else:
        effects = {arm: (values["instruction"].get(instruction) if instruction else values["all"])
                   for arm, values in report["arms"].items()}
        effects = {arm: value for arm, value in effects.items() if value is not None}
    if not effects:
        print("Нет измерений для выбранной группы.")
        return
    names, values = list(effects), list(effects.values())
    delta = [value["delta_pp"] for value in values]
    fig, ax = plt.subplots(figsize=(9, max(3, 0.36 * len(names))))
    y = np.arange(len(names))
    for position, mean, value in zip(y, delta, values):
        lo, hi = value["cluster_bootstrap_ci95_pp"]
        ax.hlines(position, lo, hi, color="C0")
        ax.scatter(mean, position, color="C0")
    ax.set_yticks(y, names)
    ax.axvline(0, color="grey", linewidth=1)
    ax.set_xlabel("Изменение успеха, п.п. (95% интервал по reset-группам)")
    ax.invert_yaxis()
    plt.show()

def plot_holdouts(report):
    counts = {}
    for row in report["rows"]:
        key = (row["skill"], row["split"])
        counts[key] = counts.get(key, 0) + row["n"]
    groups = sorted({split for _, split in counts})
    skills = sorted({skill for skill, _ in counts})
    fig, ax = plt.subplots(figsize=(9, 4))
    x = np.arange(len(groups))
    for i, skill in enumerate(skills):
        ax.bar(x + 0.35 * i, [counts.get((skill, split), 0) for split in groups],
               width=0.34, label=skill)
    ax.set_xticks(x + 0.175, groups, rotation=20)
    ax.set_ylabel("Число сцен")
    ax.legend()
    plt.show()


def plot_timeline(summary):
    points = sorted(summary["timeline"], key=lambda row: row["order"])
    instructions = sorted({name for row in points for name in row["success"]})
    fig, ax = plt.subplots(figsize=(10, 4))
    for name in instructions:
        ax.plot([row["id"] for row in points],
                [100 * row["success"].get(name, np.nan) for row in points],
                "o-", label=name)
    ax.set(xlabel="Чекпоинт в порядке серии", ylabel="Успех, %", title=summary["series"])
    if instructions:
        ax.legend()
    plt.show()

In [ ]:
# Для интерактивного просмотра измените значения и повторите запуск ячейки.
PARTITION = "ordinary_test"  # bearing_holdout, pair_holdout, joint_holdout
DENOISING_STEP = "first"  # last
INSTRUCTION = None  # move_to_object, put_into_plate, put_into_bowl
DEPTH_SKILL = None  # move_to или placement для графика поздние − ранние

for name, report in reports.items():
    print(name)
    if "comparisons" in report:
        plot_direction(report, PARTITION, DENOISING_STEP)
    elif "linear_cka" in report:
        plot_cka(report, DENOISING_STEP)
    elif "series_summary" in report:
        plot_timeline(report["series_summary"])
    elif "late_minus_early" in report:
        plot_paired(report, instruction=INSTRUCTION, depth=DEPTH_SKILL)
    elif "rows" in report and "holdout_quadrant" in report:
        plot_holdouts(report)